## Data Cleaning


### Objectives
- identify and handle duplicate records
- assess and treat missing values
- cast columns where needed
- validate key quality checks after cleaning
- prepare the dataset for exploratory analysis


In [1]:
import pandas as pd
import pyarrow
import os

In [2]:
# Load the dataset and create a working copy
data_dir = '/mnt/d/data/python/archive'
df = pd.read_csv(data_dir + '/Global_Nike.csv')
df_clean = df.copy()


In [3]:
dfs={}
for i in os.listdir(data_dir):
    if i.startswith('Nike') and i.endswith('.csv'):
        file_name = os.path.splitext(i)[0]
        dfs['df_'+file_name] = pd.read_csv(os.path.join(data_dir,i))

### Duplicate checks


In [4]:
print(df_clean.duplicated().sum())

0


In [5]:
print(df_clean.duplicated(subset=['product_id', 'country_code', 'nike_size']).sum())

0


In [6]:
### adding country column 
import pycountry

def code_to_country(code):
    country = pycountry.countries.get(alpha_2=code)
    return country.name if country else None

df_clean['country_name'] = df_clean['country_code'].apply(code_to_country)

In [7]:
df_clean['gender_segment'] = df_clean['gender_segment'].replace(
    {'WOMEN|MEN':'MEN|WOMEN',
    'GIRLS|BOYS':'BOYS|GIRLS'}
)

In [8]:
dfs.keys()

dict_keys(['df_Nike_AT', 'df_Nike_AU', 'df_Nike_BE', 'df_Nike_BG', 'df_Nike_CA', 'df_Nike_CH', 'df_Nike_CN', 'df_Nike_CZ', 'df_Nike_DE', 'df_Nike_DK', 'df_Nike_EG', 'df_Nike_ES', 'df_Nike_FI', 'df_Nike_FR', 'df_Nike_GB', 'df_Nike_GR', 'df_Nike_HR', 'df_Nike_HU', 'df_Nike_ID', 'df_Nike_IE', 'df_Nike_IL', 'df_Nike_IN', 'df_Nike_IT', 'df_Nike_JP', 'df_Nike_KR', 'df_Nike_LU', 'df_Nike_MX', 'df_Nike_MY', 'df_Nike_NL', 'df_Nike_NO', 'df_Nike_NZ', 'df_Nike_PH', 'df_Nike_PL', 'df_Nike_PT', 'df_Nike_RO', 'df_Nike_SE', 'df_Nike_SG', 'df_Nike_SI', 'df_Nike_SK', 'df_Nike_TH', 'df_Nike_TR', 'df_Nike_TW', 'df_Nike_US', 'df_Nike_VN', 'df_Nike_ZA'])

### Missing values


In [9]:
# Drop columns with no usable records for analysis
df_clean = df_clean.drop(columns=['size_count', 'available_size_count', 'employee_price', 'gtin'])


In [10]:
# Allowed null column:
# sale_price_local

In [11]:
def filling_helper(df: pd.DataFrame, *column_names: str, target_column: str):
    """
    Fill missing values in a target column only when the grouping key
    has exactly one non-null unique value for that column.
    """
    unique_counts = df.groupby(list(column_names))[target_column].nunique(dropna=True)
    valid_groups = unique_counts[unique_counts == 1].index

    reference = (
        df.dropna(subset=[target_column])
          .groupby(list(column_names))[target_column]
          .first()
          .loc[valid_groups]
    )

    row_keys = list(zip(*(df[k] for k in column_names)))
    filling_values = pd.Series(row_keys, index=df.index).map(reference)

    mask = df[target_column].isna()
    df.loc[mask, target_column] = filling_values[mask]

def availability(df: pd.DataFrame, *column_names: str, target_column: str):
    """
    Check whether a group key is reliable enough to fill a target column.
    """
    return (
        df.groupby(list(column_names))[target_column]
          .nunique(dropna=True)
          .value_counts()
          .sort_index()
    )


In [12]:
availability(df_clean,'product_id','sku',target_column='gender_segment')

gender_segment
0      1697
1    352933
2        39
Name: count, dtype: int64

In [13]:
filling_helper(df_clean,'product_id','sku',target_column='gender_segment')

In [14]:
missing_rate = (
    df_clean.groupby('country_code')['gender_segment']
    .apply(lambda x: x.isna().mean())
    .sort_values(ascending=False)
)
missing_rate

country_code
EG    1.000000
NZ    0.009993
CN    0.009774
JP    0.005958
CA    0.005396
BG    0.005186
ES    0.005137
SK    0.005062
PH    0.005007
IT    0.004931
HR    0.004907
RO    0.004890
CH    0.004883
ZA    0.004855
BE    0.004757
VN    0.004723
SI    0.004683
DE    0.004669
CZ    0.004635
SE    0.004621
IE    0.004607
NL    0.004552
LU    0.004550
GR    0.004531
TR    0.004528
AT    0.004515
PT    0.004459
US    0.004360
DK    0.004293
HU    0.004254
GB    0.004250
MY    0.004203
FI    0.004089
IL    0.003959
MX    0.003884
SG    0.003875
PL    0.003700
ID    0.003422
FR    0.003398
TW    0.003375
KR    0.002494
TH    0.002462
NO    0.000976
AU    0.000000
IN    0.000000
Name: gender_segment, dtype: float64

In [15]:
# df_clean.drop(
#     df_clean[
#         (df_clean['gender_segment'].isna()) & (df_clean['country_code'] != 'EG')
#     ].index,
#     inplace=True
# )

In [16]:
print(df_clean['gender_segment'].isnull().sum())

6656


In [17]:
availability(df_clean,'model_number','product_id','sku','product_name','category',target_column='subcategory')

subcategory
0         22
1     322542
2      45649
3      37213
4       7007
5       5173
6       5756
7       3953
8       2998
9       3725
10      4176
11      5285
12      6058
13      5838
Name: count, dtype: int64

In [18]:
availability(df_clean,'product_id','sku',target_column='brand_name')

brand_name
0        28
1    354636
2         5
Name: count, dtype: int64

In [19]:
filling_helper(df_clean,'product_id','sku',target_column='brand_name')

In [20]:
availability(df_clean,'product_id',target_column='sku')

sku
1      1409
2       381
3       896
4      1708
5      5033
       ... 
131       1
132       2
134       1
141       2
148       1
Name: count, Length: 124, dtype: int64

In [21]:
availability(df_clean,'product_id','sku',target_column='color_name')

color_name
0         47
1     253014
2      31024
3      33765
4       4365
5       1849
6       1810
7       1498
8       1466
9       2186
10      2606
11      4982
12      5500
13     10557
Name: count, dtype: int64

In [22]:
availability(df_clean,'product_id','sku','country_code',target_column='localized_size')

localized_size
0       3370
1    1444423
Name: count, dtype: int64

In [23]:
filling_helper(df_clean,'product_id','sku','country_code',target_column='localized_size')

In [24]:
availability(df_clean,'product_id','sku','country_code',target_column='size_conversion_id')

size_conversion_id
0      36818
1    1410975
Name: count, dtype: int64

In [25]:
filling_helper(df_clean,'product_id','sku','country_code',target_column='size_conversion_id')

In [26]:
availability(df_clean,'product_id','sku','model_number',target_column='sport_tags')

sport_tags
0       640
1    353792
2       237
Name: count, dtype: int64

### Casting


In [27]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1447795 entries, 0 to 1447794
Data columns (total 32 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   snapshot_date          1447795 non-null  str    
 1   country_code           1447795 non-null  str    
 2   product_name           1447795 non-null  str    
 3   model_number           1447795 non-null  str    
 4   currency               1447795 non-null  str    
 5   price_local            1447795 non-null  float64
 6   sale_price_local       210687 non-null   float64
 7   gender_segment         1441139 non-null  str    
 8   size_label             1447793 non-null  str    
 9   category               1447795 non-null  str    
 10  subcategory            1447767 non-null  str    
 11  product_id             1447795 non-null  str    
 12  sku                    1447793 non-null  str    
 13  style_color            1447795 non-null  str    
 14  brand_name             144747

In [29]:
df_clean['snapshot_date'] = pd.to_datetime(df_clean['snapshot_date'])

### Validation


In [4]:
print((df_clean['price_local'] < 0).sum())


0


In [33]:
# Current price should not be higher than the original pre-discount price
print((df_clean['price_local'] > df_clean['sale_price_local']).sum())


0


In [34]:
df_clean[['price_local', 'sale_price_local', 'discount_pct']].dropna().head(10)

,price_local,sale_price_local,discount_pct
50,21.97,30.0,26.7667
51,21.97,30.0,26.7667
52,21.97,30.0,26.7667
53,21.97,30.0,26.7667
54,21.97,30.0,26.7667
55,50.97,92.0,44.5978
56,50.97,92.0,44.5978
57,50.97,92.0,44.5978
58,50.97,92.0,44.5978
59,50.97,92.0,44.5978


In [35]:
df_clean['gender_segment'].value_counts()

gender_segment
MEN                     579987
WOMEN                   373417
MEN|WOMEN               231884
BOYS|GIRLS              193003
GIRLS                    32659
BOYS                     27176
MEN|BOYS|WOMEN|GIRLS      2629
WOMEN|GIRLS                253
BOYS|WOMEN|GIRLS           110
MEN|BOYS|GIRLS              14
MEN|WOMEN|GIRLS              7
Name: count, dtype: int64

In [36]:
df_clean['category'].value_counts().head(20)

category
FOOTWEAR              724811
APPAREL               696577
EQUIPMENT              26389
PHYSICAL_GIFT_CARD        16
DIGITAL_GIFT_CARD          2
Name: count, dtype: int64

In [37]:
df_clean['availability_level'].value_counts()

availability_level
LOW       496767
HIGH      493247
OOS       370116
MEDIUM     87663
Name: count, dtype: int64

In [38]:
df_clean['country_code'].nunique()

45

In [39]:
df['country_code'].nunique()

45

In [40]:
set(df['country_code'].dropna().unique()) - set(df_clean['country_code'].dropna().unique())

set()

In [41]:
df.loc[df['country_code'].isin(['EG', 'NZ'])]

,snapshot_date,country_code,product_name,model_number,currency,price_local,sale_price_local,gender_segment,size_label,category,...,canonical_url,image_url,gtin,stock_keeping_unit_id,catalog_sku_id,nike_size,localized_size,size_conversion_id,sport_tags,record_source
1444021,2026-03-19,NZ,Air Jordan 1 Elevate Low,DH7004,NZD,220.0,NaN,WOMEN,10,FOOTWEAR,...,https://www.nike.com/nz/t/air-jordan-1-elevate...,https://secure-images.nike.com/is/image/DotCom...,NaN,30366278.0,19b89cfb-d4f5-3226-8b2e-b5d38ce2881d,10,NaN,b6635f29-bb01-3ccd-9ade-dae66cf82e4c,Lifestyle,thread_exact
1444022,2026-03-19,NZ,Air Jordan 1 Elevate Low,DH7004,NZD,220.0,NaN,WOMEN,10.5,FOOTWEAR,...,https://www.nike.com/nz/t/air-jordan-1-elevate...,https://secure-images.nike.com/is/image/DotCom...,NaN,30366273.0,50d0c961-f25c-35fd-88bc-e4b726492957,10.5,NaN,7702dda3-62ca-3410-b4f3-97025be1089d,Lifestyle,thread_exact
1444023,2026-03-19,NZ,Air Jordan 1 Elevate Low,DH7004,NZD,220.0,NaN,WOMEN,11,FOOTWEAR,...,https://www.nike.com/nz/t/air-jordan-1-elevate...,https://secure-images.nike.com/is/image/DotCom...,NaN,30366279.0,84247c6d-6e30-3884-b17b-00d7029a3ea7,11,NaN,7a1af8aa-7163-34a2-bf12-c2c7bc01b85c,Lifestyle,thread_exact
1444024,2026-03-19,NZ,Air Jordan 1 Elevate Low,DH7004,NZD,220.0,NaN,WOMEN,11.5,FOOTWEAR,...,https://www.nike.com/nz/t/air-jordan-1-elevate...,https://secure-images.nike.com/is/image/DotCom...,NaN,30366274.0,f315a6ef-d1f6-3cf4-b95c-3cf2cdb74df9,11.5,NaN,539c6c43-d060-30b0-be5b-47c47f7c7658,Lifestyle,thread_exact
1444025,2026-03-19,NZ,Air Jordan 1 Elevate Low,DH7004,NZD,220.0,NaN,WOMEN,5,FOOTWEAR,...,https://www.nike.com/nz/t/air-jordan-1-elevate...,https://secure-images.nike.com/is/image/DotCom...,NaN,30366272.0,c61f9656-6254-3246-91c3-089cba0eace2,5,NaN,fd811e2b-30de-3e11-a2c4-603994ea42e4,Lifestyle,thread_exact
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1447790,2026-03-19,EG,Nike Dry 'Oregon Project',863190,EGP,849.0,NaN,NaN,2XL,APPAREL,...,https://www.nike.com/eg/t/dry-oregon-project-r...,https://secure-images.nike.com/is/image/DotCom...,NaN,18492928.0,c672d714-7178-329d-946c-271b21d4ef1d,2XL,NaN,NaN,NaN,thread_exact
1447791,2026-03-19,EG,Nike Dry 'Oregon Project',863190,EGP,849.0,NaN,NaN,L,APPAREL,...,https://www.nike.com/eg/t/dry-oregon-project-r...,https://secure-images.nike.com/is/image/DotCom...,NaN,19424137.0,a7f47e94-dd5e-3610-b993-982d0d7e9db6,L,NaN,NaN,NaN,thread_exact
1447792,2026-03-19,EG,Nike Dry 'Oregon Project',863190,EGP,849.0,NaN,NaN,M,APPAREL,...,https://www.nike.com/eg/t/dry-oregon-project-r...,https://secure-images.nike.com/is/image/DotCom...,NaN,19424138.0,9b463e62-0d3e-32ee-ac1c-3877f4d36883,M,NaN,NaN,NaN,thread_exact
1447793,2026-03-19,EG,Nike Dry 'Oregon Project',863190,EGP,849.0,NaN,NaN,S,APPAREL,...,https://www.nike.com/eg/t/dry-oregon-project-r...,https://secure-images.nike.com/is/image/DotCom...,NaN,19424139.0,bc802f14-7de1-3e0e-a418-b2444580daa9,S,NaN,NaN,NaN,thread_exact


### Final observations

- Missing values were handled conservatively: columns were filled only when the key was consistent, otherwise rows were dropped when the field was required for later analysis.
- `snapshot_date` was converted to datetime, and `stock_keeping_unit_id` was cast to integer after validation.
- Validation checks suggest the cleaned dataset is ready for exploratory analysis.


### Export cleaned dataset


In [42]:
df_clean.to_csv('nike_cleaned.csv', index=False)

In [43]:
df_clean.to_parquet('/mnt/d/data/python/archive/nike_cleaned.parquet', index=False)

### Final observations on missing values

- Missing values were explored both globally and by country to distinguish between random row-level gaps and structural missingness.
- Four columns were found to be completely empty (`size_count`, `available_size_count`, `employee_price`, and `gtin`), so they were identified as non-informative and suitable for removal.
- For low-missing columns, key-based filling was tested using only reliable groups with exactly one non-null unique value per key.
- `gender_segment` was examined carefully using `(product_id, sku)` as a reference key. This showed that most groups had a single consistent value, but a small number had conflicting values, so filling was restricted to safe cases only.
- Country-level inspection showed that Egypt (`EG`) had complete missingness in `gender_segment`, while all other countries had only very small missing rates. Because dropping `EG` would remove an entire market, its missing values were retained rather than imputed from weak country-level assumptions.
- Partial missing `gender_segment` values in other countries were considered less problematic because their rates were very low, but dropping such rows would also remove information from other columns. Therefore, row removal is more appropriate for analysis-specific subsets than for the main cleaned dataset.
- `brand_name` showed strong consistency under `(product_id, sku)`, making it a good candidate for safe filling.
- `localized_size` and `size_conversion_id` also showed strong consistency when checked with `(product_id, sku, country_code)`, so they were explored as fillable columns under strict key conditions.
- `color_name` was not stable enough under the tested key, so broad imputation would be risky.
- `sport_tags` showed mixed behavior: many groups were stable, but some had multiple values, so leaving missing values may be more defensible than forcing imputation.
- Overall, missing-value treatment followed a conservative rule: fill only when the key shows a unique and stable mapping; otherwise keep the missing value or exclude it only in analysis-specific datasets.
